# Fine-tune GPT-2 Vietnamese for Math Word Problems - Exp15 Standalone

**Goal:** fine-tune the local `nlphustgpt2-vietnamese` model to answer Vietnamese math word problems by generating only the final numeric answer.

**Inputs:** `dataset-math/train.json`, `dataset-math/valid.json`, and local base model folder `nlphustgpt2-vietnamese`.

**Pipeline:** load data -> oversample GSM records -> numeric-answer filtering -> answer-only SFT training -> beam-search validation generation -> relative-error scoring.

**Important:** this notebook is standalone. All helper functions needed for data processing, training, generation, and evaluation are defined inside this notebook.


In [2]:
import os, sys, json, math, time, re, random, hashlib, inspect
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


Torch: 2.11.0+cu130
CUDA: True | GPU count: 1
0 NVIDIA GeForce RTX 5060


In [3]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Khong tim thay path nao: " + " | ".join(map(str, paths)))

ROOT = Path.cwd()
DATA_DIR = first_existing(ROOT / "dataset-math", "dataset-math")
MODEL_NAME = str(first_existing(ROOT / "nlphustgpt2-vietnamese", "nlphustgpt2-vietnamese"))

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"

EXP_DIR = ROOT / "experiments_12h" / "exp_15_gsm_oversample_epoch5"
ARTIFACT_DIR = EXP_DIR / "artifacts"
MODEL_DIR = ARTIFACT_DIR / "model"
VALID_OUTPUT_PATH = ARTIFACT_DIR / "valid_output.json"
VALID_REPORT_PATH = ARTIFACT_DIR / "valid_report.json"
SUMMARY_PATH = ARTIFACT_DIR / "summary.json"

SAFE_EOS_ID = 50256
SAFE_EOS_DECODED_FALLBACKS = ["hue", "<|endoftext|>"]

CFG = {
    "name": "exp_15_gsm_oversample_epoch5",
    "model_name": MODEL_NAME,
    "train_file": str(TRAIN_FILE),
    "valid_file": str(VALID_FILE),
    "seed": 42,
    "drop_unextractable": True,
    "require_numeric_answer": True,
    "dataloader_num_workers": 0,
    "logging_steps": 100,
    "eval_strategy": "epoch",
    "save_strategy": "no",
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "lr_scheduler_type": "cosine",
    "target_mode": "answer_norm",
    "prompt_style": "answer",
    "output_style": "answer_prefix",
    "include_type": True,
    "instruction": "Hãy giải bài toán. Chỉ viết kết quả số cuối cùng, không viết lời giải.",
    "oversample_type_multipliers": {
        "GSM_AnsAug": 2,
        "GSM_FOBAR": 2,
        "GSM_Rephrased": 2,
        "GSM_SV": 2,
    },
    "epochs": 5,
    "max_length": 384,
    "per_device_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "grad_accum": 8,
    "lr": 3e-5,
    "generation": {
        "max_new_tokens": 8,
        "num_beams": 3,
        "do_sample": False,
        "repetition_penalty": 1.05,
    },
}

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print(json.dumps(CFG, ensure_ascii=False, indent=2))


DATA_DIR: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/dataset-math
MODEL_NAME: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/nlphustgpt2-vietnamese
ARTIFACT_DIR: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/experiments_12h/exp_15_gsm_oversample_epoch5/artifacts
{
  "name": "exp_15_gsm_oversample_epoch5",
  "model_name": "/mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/nlphustgpt2-vietnamese",
  "train_file": "/mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/dataset-math/train.json",
  "valid_file": "/mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/dataset-math/valid.json",
  "seed": 42,
  "drop_unextractable": true,
  "require_numeric_answer": true,
  "dataloader_num_workers": 0,
  "logging_steps": 100,
  "eval_strategy": "epoch",
  "save_strategy": "no",
  "warmup_ratio": 0.03,
  "weight_decay": 0.0,
  "lr_scheduler_type": "cosine",
  "target_mode": "answer_norm",
  "prompt_style": "answer",
  "output_style": "answer_prefix",
  "include

In [4]:
# ============================================================
# 2. Reproducibility and file utilities
# ============================================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_records(path: str | Path) -> list[dict]:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    if not dir_path.exists():
        return ""
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()


def save_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


seed_everything(CFG["seed"])


In [5]:
# ============================================================
# 3. Answer extraction and numeric normalization
# ============================================================
VI_ANCHORS = [
    r"Câu trả lời là\s*[:：]?",
    r"Đáp án là\s*[:：]?",
    r"Đáp án\s*[:：]",
]
EN_ANCHORS = [
    r"The answer is\s*[:：]?",
    r"####",
]
BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}


def clean_answer_text(answer: str | None) -> str | None:
    if answer is None:
        return None
    ans = str(answer).strip()
    ans = re.sub(r"^(?:là|=)\s*", "", ans, flags=re.IGNORECASE).strip()
    ans = ans.strip(" .$。、、,")
    ans = re.sub(r"\s+", " ", ans)
    return ans or None


def extract_answer(text: str | None, anchors: list[str]) -> str | None:
    if not text:
        return None
    for anc in anchors:
        matches = list(re.finditer(anc, text))
        if matches:
            m = matches[-1]
            tail = text[m.end():].strip().split("\n")[0]
            return clean_answer_text(tail)
    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_text(boxes[-1])
    return None


def extract_gold(rec: dict) -> str | None:
    return extract_answer(rec.get("response_vi"), VI_ANCHORS)


def extract_pred(rec: dict) -> str | None:
    return extract_answer(rec.get("model_output"), VI_ANCHORS + EN_ANCHORS)


def parse_number(s: str | None) -> float | None:
    if s is None:
        return None
    t = s.strip()
    if not t:
        return None

    if re.fullmatch(r"-?\d+,\d+", t):
        try:
            return float(t.replace(",", "."))
        except ValueError:
            return None

    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", t):
        try:
            val = float(t)
            return val if math.isfinite(val) else None
        except ValueError:
            return None

    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in (r"\,", r"\!", r"\;", r"\ ", r"\left", r"\right"):
        t = t.replace(token, "")
    for token in (r"\cdot", r"\times"):
        t = t.replace(token, "*")

    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace(r"\pi", "pi")

    t = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", t)

    has_period = "." in t
    n_commas = t.count(",")
    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)
    if not t or "," in t:
        return None

    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    t = t.replace("^", "**")
    try:
        val = eval(t, {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(val, bool):
        return None
    if isinstance(val, (int, float)):
        val = float(val)
        return val if math.isfinite(val) else None
    return None


def normalize_answer(answer: str | None) -> str | None:
    ans = clean_answer_text(answer)
    val = parse_number(ans)
    if val is None:
        return ans
    if abs(val - round(val)) < 1e-9 and abs(val) < 1e16:
        return str(int(round(val)))
    return f"{val:.12g}"


In [6]:
# ============================================================
# 4. Prompt, target, dataset, and collator
# ============================================================
def build_prompt(rec: dict, cfg: dict) -> str:
    lines = []
    if cfg.get("instruction"):
        lines.append(str(cfg["instruction"]).strip())
    if cfg.get("include_type"):
        lines.append(f"Dạng bài: {rec.get('type', '')}")
    lines.append(f"Câu hỏi: {rec['query_vi'].strip()}")
    lines.append("Đáp án là: ")
    return "\n".join(lines)


def build_target(rec: dict, cfg: dict, idx: int = 0) -> str | None:
    answer_raw = extract_gold(rec)
    answer_norm = normalize_answer(answer_raw)
    answer = answer_norm if "norm" in cfg.get("target_mode", "answer_norm") else clean_answer_text(answer_raw)
    return answer or None


def apply_train_sampling(records: list[dict], cfg: dict) -> list[dict]:
    multipliers = cfg.get("oversample_type_multipliers") or {}
    if not multipliers:
        return records

    expanded = []
    for rec in records:
        multiplier = int(multipliers.get(rec.get("type"), 1))
        if multiplier > 0:
            expanded.extend([rec] * multiplier)

    if not expanded:
        raise ValueError("oversample_type_multipliers removed all training records")

    counts = Counter(rec.get("type") for rec in expanded)
    print(
        f"train sampling: original={len(records)} expanded={len(expanded)} "
        f"type_counts={dict(sorted(counts.items()))}",
        flush=True,
    )
    return expanded


class SFTDataset(Dataset):
    def __init__(self, records: list[dict], tokenizer, max_length: int, cfg: dict, name: str):
        self.tok = tokenizer
        self.max_length = max_length
        self.cfg = cfg
        self.examples = []
        skipped = 0
        skipped_non_numeric = 0

        for idx, rec in enumerate(records):
            answer_raw = extract_gold(rec)
            if cfg.get("require_numeric_answer") and parse_number(answer_raw) is None:
                skipped_non_numeric += 1
                continue

            target = build_target(rec, cfg, idx)
            if not target:
                skipped += 1
                if cfg.get("drop_unextractable", True):
                    continue
                target = "0"
            self.examples.append((rec, target))

        if not self.examples:
            raise ValueError(f"{name}: no usable examples")

        print(
            f"{name}: usable={len(self.examples)} | "
            f"skipped_no_answer={skipped} | skipped_non_numeric={skipped_non_numeric}",
            flush=True,
        )

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, i: int) -> Dict[str, List[int]]:
        rec, target = self.examples[i]
        prompt = build_prompt(rec, self.cfg)
        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        t_ids = self.tok(target, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        if len(t_ids) >= self.max_length:
            t_ids = t_ids[-self.max_length:]
            p_ids = []
        else:
            p_ids = p_ids[-(self.max_length - len(t_ids)):]

        ids = p_ids + t_ids
        labels = [-100] * len(p_ids) + t_ids
        ids = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]
        return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}


@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


In [7]:
# ============================================================
# 5. Scoring and evaluation
# ============================================================
def clean_generated_answer(text: str | None) -> str:
    if not text:
        return ""
    t = text.strip()
    for eos_text in SAFE_EOS_DECODED_FALLBACKS:
        if eos_text and t.endswith(eos_text):
            t = t[: -len(eos_text)].strip()
    extracted = extract_answer(t, VI_ANCHORS + EN_ANCHORS)
    if extracted:
        t = extracted.strip()
    t = t.split("\n")[0].strip()
    for marker in ["Câu hỏi:", "Lời giải:", "Question:", "Answer:"]:
        if marker in t:
            t = t.split(marker)[0].strip()
    t = re.split(r"\s+(?:Đáp án|Câu trả lời)\s+là\s*[:：]?", t)[0].strip()
    return t.strip(" .$。、、,")


def format_model_output(raw_text: str, cfg: dict) -> str:
    t = raw_text.strip()
    for eos_text in SAFE_EOS_DECODED_FALLBACKS:
        if eos_text and t.endswith(eos_text):
            t = t[: -len(eos_text)].strip()
    return "Đáp án là: " + clean_generated_answer(t)


def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))


def score_one(re_val: float | None, extractable: bool) -> int:
    if not extractable or re_val is None:
        return 0
    if re_val <= 0.01:
        return 10
    if re_val <= 0.10:
        return 5
    if re_val <= 0.50:
        return 1
    return 0


def evaluate(pred_items: list[dict], gold_items: list[dict]) -> dict:
    if len(pred_items) != len(gold_items):
        raise ValueError(f"Prediction count {len(pred_items)} != gold count {len(gold_items)}")

    rows = []
    total = extractable = numeric_pairs = 0
    buckets = {"10": 0, "5": 0, "1": 0, "0": 0}
    rel_errors = []
    by_type = {}

    for pred_rec, gold_rec in zip(pred_items, gold_items):
        gold_ans = extract_gold(gold_rec)
        pred_ans = extract_pred(pred_rec)
        is_extractable = pred_ans is not None
        extractable += int(is_extractable)

        gold_num = parse_number(gold_ans)
        pred_num = parse_number(pred_ans)
        re_val = rel_error(pred_num, gold_num)
        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s
        buckets[str(s)] += 1
        typ = gold_rec.get("type") or pred_rec.get("type")
        by_type.setdefault(typ, {"n": 0, "raw_score": 0, "bucket0": 0})
        by_type[typ]["n"] += 1
        by_type[typ]["raw_score"] += s
        by_type[typ]["bucket0"] += int(s == 0)

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": typ,
            "gold_answer": gold_ans,
            "pred_answer": pred_ans,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    for typ, item in by_type.items():
        item["score_10"] = item["raw_score"] / item["n"] if item["n"] else 0.0

    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": n * 10,
            "score_10": total / n if n else 0.0,
            "score_pct": total / (n * 10) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
            "by_type": by_type,
        },
        "rows": rows,
    }


In [8]:
# ============================================================
# 6. Generation
# ============================================================
def generate_outputs(model_path: str | Path, records: list[dict], output_path: Path, cfg: dict) -> list[dict]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    tokenizer.pad_token_id = SAFE_EOS_ID
    tokenizer.eos_token_id = SAFE_EOS_ID

    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=dtype,
        local_files_only=True,
    ).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()

    outputs = []
    max_length = int(cfg.get("max_length", 384))
    generation = cfg.get("generation", {})
    gen_kwargs = {
        "max_new_tokens": int(generation.get("max_new_tokens", 8)),
        "do_sample": bool(generation.get("do_sample", False)),
        "num_beams": int(generation.get("num_beams", 3)),
        "repetition_penalty": float(generation.get("repetition_penalty", 1.05)),
        "pad_token_id": SAFE_EOS_ID,
        "eos_token_id": SAFE_EOS_ID,
    }

    vocab_n = model.transformer.wte.num_embeddings
    with torch.inference_mode():
        for rec in tqdm(records, desc="Generating"):
            prompt = build_prompt(rec, cfg)
            enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(device)
            ids = enc["input_ids"].clamp(max=vocab_n - 1)
            gen = model.generate(input_ids=ids, attention_mask=enc.get("attention_mask"), **gen_kwargs)
            raw_text = tokenizer.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
            outputs.append({
                "id": len(outputs),
                "query_vi": rec["query_vi"],
                "type": rec.get("type"),
                "model_output": format_model_output(raw_text, cfg),
            })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    save_json(output_path, outputs)
    Path(str(output_path) + ".sha256.txt").write_text(sha256_file(output_path) + "\n", encoding="utf-8")
    del model
    torch.cuda.empty_cache()
    return outputs


In [9]:
# ============================================================
# 7. Load data and apply GSM oversampling
# ============================================================
train_records_raw = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)
train_records = apply_train_sampling(train_records_raw, CFG)

print("Raw train records:", len(train_records_raw))
print("Expanded train records:", len(train_records))
print("Validation records:", len(valid_records))
print("Raw train type counts:", dict(sorted(Counter(r.get("type") for r in train_records_raw).items())))
print("Expanded train type counts:", dict(sorted(Counter(r.get("type") for r in train_records).items())))


train sampling: original=100000 expanded=160738 type_counts={'GSM_AnsAug': 40658, 'GSM_FOBAR': 20382, 'GSM_Rephrased': 40596, 'GSM_SV': 19840, 'MATH_AnsAug': 18991, 'MATH_FOBAR': 3769, 'MATH_Rephrased': 12788, 'MATH_SV': 3714}
Raw train records: 100000
Expanded train records: 160738
Validation records: 1000
Raw train type counts: {'GSM_AnsAug': 20329, 'GSM_FOBAR': 10191, 'GSM_Rephrased': 20298, 'GSM_SV': 9920, 'MATH_AnsAug': 18991, 'MATH_FOBAR': 3769, 'MATH_Rephrased': 12788, 'MATH_SV': 3714}
Expanded train type counts: {'GSM_AnsAug': 40658, 'GSM_FOBAR': 20382, 'GSM_Rephrased': 40596, 'GSM_SV': 19840, 'MATH_AnsAug': 18991, 'MATH_FOBAR': 3769, 'MATH_Rephrased': 12788, 'MATH_SV': 3714}


In [10]:
# ============================================================
# 8. Quick offline model check
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)

tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print(type(tokenizer))
print(type(model))
print("vocab_size:", model.config.vocab_size)

sample_prompt = build_prompt(valid_records[0], CFG)
print("\nSample prompt:\n", sample_prompt)
print("Gold answer:", normalize_answer(extract_gold(valid_records[0])))

del model
torch.cuda.empty_cache()


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 22752.52it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
vocab_size: 50257

Sample prompt:
 Hãy giải bài toán. Chỉ viết kết quả số cuối cùng, không viết lời giải.
Dạng bài: GSM_Rephrased
Câu hỏi: Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?
Đáp án là: 
Gold answer: 37


In [11]:
# ============================================================
# 9. Train
# ============================================================
seed_everything(CFG["seed"])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID
model.gradient_checkpointing_enable()
model.config.use_cache = False

max_length = int(CFG["max_length"])
train_ds = SFTDataset(train_records, tokenizer, max_length, CFG, name="train")
valid_ds = SFTDataset(valid_records, tokenizer, max_length, CFG, name="valid")
collator = PadCollator(pad_id=SAFE_EOS_ID)

per_device_bs = int(CFG["per_device_batch_size"])
grad_accum = int(CFG["grad_accum"])
epochs = float(CFG["epochs"])
effective_batch = per_device_bs * grad_accum * max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_ds) / effective_batch)
warmup_steps = int(steps_per_epoch * epochs * float(CFG["warmup_ratio"]))

print(
    f"samples={len(train_ds)} epochs={epochs} bs={per_device_bs} "
    f"accum={grad_accum} eff={effective_batch} steps/epoch={steps_per_epoch} "
    f"total_steps={steps_per_epoch * epochs:.0f} warmup_steps={warmup_steps}",
    flush=True,
)

ta_kwargs = dict(
    output_dir=str(MODEL_DIR),
    num_train_epochs=epochs,
    per_device_train_batch_size=per_device_bs,
    per_device_eval_batch_size=int(CFG.get("per_device_eval_batch_size", per_device_bs)),
    gradient_accumulation_steps=grad_accum,
    learning_rate=float(CFG["lr"]),
    warmup_steps=warmup_steps,
    lr_scheduler_type=CFG["lr_scheduler_type"],
    weight_decay=float(CFG["weight_decay"]),
    fp16=torch.cuda.is_available(),
    logging_steps=int(CFG["logging_steps"]),
    save_strategy=CFG["save_strategy"],
    save_total_limit=1,
    report_to="none",
    seed=int(CFG["seed"]),
    dataloader_num_workers=int(CFG["dataloader_num_workers"]),
    remove_unused_columns=False,
)

sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    ta_kwargs["eval_strategy"] = CFG["eval_strategy"]
else:
    ta_kwargs["evaluation_strategy"] = CFG["eval_strategy"]

trainer = Trainer(
    model=model,
    args=TrainingArguments(**ta_kwargs),
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
)

t0 = time.time()
trainer.train()
train_seconds = time.time() - t0
print(f"Train seconds: {train_seconds:.1f}")

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
(MODEL_DIR / "model_hash.txt").write_text(sha256_dir(MODEL_DIR) + "\n", encoding="utf-8")

# Release VRAM before validation generation.
del trainer, model
torch.cuda.empty_cache()


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 18765.33it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train: usable=156466 | skipped_no_answer=0 | skipped_non_numeric=4272
valid: usable=965 | skipped_no_answer=0 | skipped_non_numeric=35
samples=156466 epochs=5.0 bs=4 accum=8 eff=32 steps/epoch=4890 total_steps=24450 warmup_steps=733


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.858188,1.869943
2,1.059614,1.299049
3,0.612749,1.046889
4,0.459107,1.012306
5,0.374421,1.027916


Train seconds: 10781.1


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]


In [12]:
# ============================================================
# 10. Generate validation outputs
# ============================================================
valid_outputs = generate_outputs(MODEL_DIR, valid_records, VALID_OUTPUT_PATH, CFG)

print("\nExample output:")
print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Generating: 100%|██████████| 1000/1000 [00:16<00:00, 62.25it/s]



Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "Đáp án là: 26"
}


In [13]:
# ============================================================
# 11. Evaluate and save reports
# ============================================================
report = evaluate(valid_outputs, valid_records)
report["summary"]["train_seconds"] = train_seconds
report["summary"]["experiment"] = CFG["name"]
report["summary"]["config"] = CFG

save_json(VALID_REPORT_PATH, report)
save_json(SUMMARY_PATH, report["summary"])

print("\nFinal validation score:")
summary = report["summary"]
print(f'{summary["raw_score"]} / {summary["max_raw_score"]}  ({summary["score_pct"]*100:.2f}%)')
print(f'Score /10: {summary["score_10"]:.3f}')
print("Buckets:", summary["buckets"])
print("Extractable:", summary["extractable"])
print("Numeric pairs:", summary["numeric_pairs"])
print("\nBy type:")
print(json.dumps(summary["by_type"], ensure_ascii=False, indent=2))



Final validation score:
6012 / 10000  (60.12%)
Score /10: 6.012
Buckets: {'10': 578, '5': 19, '1': 137, '0': 266}
Extractable: 1000
Numeric pairs: 959

By type:
{
  "GSM_Rephrased": {
    "n": 197,
    "raw_score": 1235,
    "bucket0": 46,
    "score_10": 6.269035532994923
  },
  "MATH_Rephrased": {
    "n": 116,
    "raw_score": 518,
    "bucket0": 46,
    "score_10": 4.4655172413793105
  },
  "MATH_SV": {
    "n": 41,
    "raw_score": 211,
    "bucket0": 14,
    "score_10": 5.146341463414634
  },
  "GSM_AnsAug": {
    "n": 209,
    "raw_score": 1935,
    "bucket0": 10,
    "score_10": 9.258373205741627
  },
  "GSM_SV": {
    "n": 97,
    "raw_score": 232,
    "bucket0": 48,
    "score_10": 2.3917525773195876
  },
  "GSM_FOBAR": {
    "n": 122,
    "raw_score": 352,
    "bucket0": 57,
    "score_10": 2.8852459016393444
  },
  "MATH_AnsAug": {
    "n": 173,
    "raw_score": 1338,
    "bucket0": 30,
    "score_10": 7.734104046242774
  },
  "MATH_FOBAR": {
    "n": 45,
    "raw_score": 

In [14]:
# ============================================================
# 12. Inspect saved artifacts
# ============================================================
print("Model dir:", MODEL_DIR)
print("Validation output:", VALID_OUTPUT_PATH, "exists=", VALID_OUTPUT_PATH.exists())
print("Validation report:", VALID_REPORT_PATH, "exists=", VALID_REPORT_PATH.exists())
print("Summary:", SUMMARY_PATH, "exists=", SUMMARY_PATH.exists())

if SUMMARY_PATH.exists():
    saved_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    print(json.dumps({
        "score_10": saved_summary.get("score_10"),
        "buckets": saved_summary.get("buckets"),
        "extractable": saved_summary.get("extractable"),
        "numeric_pairs": saved_summary.get("numeric_pairs"),
        "train_seconds": saved_summary.get("train_seconds"),
    }, ensure_ascii=False, indent=2))


Model dir: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/experiments_12h/exp_15_gsm_oversample_epoch5/artifacts/model
Validation output: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/experiments_12h/exp_15_gsm_oversample_epoch5/artifacts/valid_output.json exists= True
Validation report: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/experiments_12h/exp_15_gsm_oversample_epoch5/artifacts/valid_report.json exists= True
Summary: /mnt/c/Users/Admin/Desktop/UET/DeepLearning/new_btl_DL/experiments_12h/exp_15_gsm_oversample_epoch5/artifacts/summary.json exists= True
{
  "score_10": 6.012,
  "buckets": {
    "10": 578,
    "5": 19,
    "1": 137,
    "0": 266
  },
  "extractable": 1000,
  "numeric_pairs": 959,
  "train_seconds": 10781.109933137894
}
